In [2]:
%%capture
!pip install easyocr opencv-python matplotlib joblib

In [ ]:
import os
import cv2
import numpy as np
import easyocr
from typing import List, Tuple, Optional, Dict
from tqdm.notebook import tqdm
from joblib import Parallel, delayed
import time
import pickle
class ImageClassifier:
    """
    Quét một phần hoặc toàn bộ thư mục ảnh, phân loại chúng thành các nhóm
    CÓ VĂN BẢN và KHÔNG CÓ VĂN BẢN, lưu kết quả ra file và trả về cả hai danh sách.
    """

    def __init__(self, input_path: str, ignore_boxes: List[Tuple], gpu: bool = True, progress: Optional[Dict[str, int]] = None, embedding_info_path: Optional[str] = None):
        # Phần khởi tạo __init__ đã được cập nhật
        self.input_path = input_path
        self.ignore_boxes = ignore_boxes
        
        print("Đang quét và sắp xếp tất cả các đường dẫn ảnh...")
        all_image_paths = []
        if not os.path.isdir(self.input_path):
            raise ValueError(f"Đường dẫn đầu vào không tồn tại hoặc không phải là thư mục: {self.input_path}")

        for folder_name in os.listdir(self.input_path):
            folder_path = os.path.join(self.input_path, folder_name)
            if os.path.isdir(folder_path):
                files = [
                    os.path.join(folder_path, file) 
                    for file in os.listdir(folder_path) 
                    if file.endswith('.jpg')
                ]
                all_image_paths.extend(files)
        
        all_image_paths.sort()
        
        # Lọc all_image_paths dựa trên embedding_info_path nếu được cung cấp
        if embedding_info_path and os.path.exists(embedding_info_path):
            print(f"Đang tải thông tin embedding từ: {embedding_info_path}")
            try:
                with open(embedding_info_path, 'rb') as f:
                    embedding_info = pickle.load(f)
                
                if 'paths' in embedding_info:
                    allowed_paths_set = set(embedding_info['paths'])
                    print(f"Tìm thấy {len(allowed_paths_set)} paths trong embedding_info")
                    
                    # Lọc all_image_paths để chỉ giữ lại những path có tồn tại trong embedding_info
                    filtered_paths = []
                    for img_path in all_image_paths:
                        # Trích xuất phần {video_name}/{frame_index}.jpg từ đường dẫn đầy đủ
                        path_parts = img_path.split(os.sep)
                        if len(path_parts) >= 2:
                            relative_path = f"{path_parts[-2]}/{path_parts[-1]}"
                            if relative_path in allowed_paths_set:
                                filtered_paths.append(img_path)
                    
                    all_image_paths = filtered_paths
                    print(f"Sau khi lọc dựa trên embedding_info, còn lại {len(all_image_paths)} ảnh")
                else:
                    print("Cảnh báo: Không tìm thấy trường 'paths' trong embedding_info")
            except Exception as e:
                print(f"Lỗi khi tải embedding_info từ {embedding_info_path}: {e}")
                print("Tiếp tục với toàn bộ danh sách ảnh...")
        elif embedding_info_path:
            print(f"Cảnh báo: File embedding_info không tồn tại: {embedding_info_path}")
            print("Tiếp tục với toàn bộ danh sách ảnh...") 
        
        total_images = len(all_image_paths)
        self.start_index = 0
        self.end_index = total_images

        if progress:
            self.start_index = progress.get('start', 0)
            self.end_index = progress.get('end', total_images) 
        print({
            "length": total_images,
            "start": self.start_index,
            "end": self.end_index
        })
        self.start_index = max(0, self.start_index)
        self.end_index = min(total_images, self.end_index)
        if self.start_index >= self.end_index:
             print(f"Cảnh báo: 'start' ({self.start_index}) lớn hơn hoặc bằng 'end' ({self.end_index}). Sẽ không có ảnh nào được xử lý.")
             self.paths_to_process = []
        else:
            self.paths_to_process = all_image_paths[self.start_index:self.end_index]

        print(f"Đã tìm thấy tổng cộng {total_images} ảnh.")
        print(f"Sẽ xử lý {len(self.paths_to_process)} ảnh từ chỉ số {self.start_index} đến {self.end_index}.")

        print("Đang khởi tạo EasyOCR Reader...")
        try:
            self.reader = easyocr.Reader(['en'], gpu=gpu, detector="craft")
            print(f"EasyOCR Reader đã sẵn sàng (GPU: {gpu}).")
        except Exception as e:
            print(f"Không thể khởi tạo EasyOCR với GPU={gpu}, lỗi: {e}. Thử lại với CPU.")
            self.reader = easyocr.Reader(['en'], gpu=False, detector="craft")
            print("EasyOCR Reader đã sẵn sàng (GPU: False).")

    # --- Các hàm trợ giúp không thay đổi ---
    def _read_image_and_path(self, path: str):
        try:
            img = cv2.imread(path)
            return (img, path)
        except Exception as e:
            print(f"💥 Lỗi ngoại lệ khi đọc ảnh {path}: {e}")
            return (None, path)

    def _preprocess_image(self, image, scale_factor=2.0, target_width=None):
        if isinstance(image, str):
            img = cv2.imread(image)
            if img is None:
                print(f"Lỗi: Không thể đọc ảnh từ đường dẫn: {image}")
                return None
        else:
            img = image

        # ignore boxes if image is landscape
        if img.shape[0] < img.shape[1]:
            for box in self.ignore_boxes:
                cv2.rectangle(img, box[0], box[1], (0), -1)
        
        # upscale image
        width = int(img.shape[1] * scale_factor)
        height = int(img.shape[0] * scale_factor)
        upscaled_image = cv2.resize(img, (width, height), interpolation=cv2.INTER_CUBIC)

        # convert BGR to gray - Sửa lỗi: dùng upscaled_image thay vì img
        if len(upscaled_image.shape) == 3 and upscaled_image.shape[2] == 3:
            gray = cv2.cvtColor(upscaled_image, cv2.COLOR_BGR2GRAY)
        else:
            gray = upscaled_image 

        # enhance gray image
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4,4))
        enhanced_gray = clahe.apply(gray)
        
        processed_image = cv2.cvtColor(enhanced_gray, cv2.COLOR_GRAY2BGR)
        
        # Padding nếu target_width được cung cấp
        if target_width is not None and processed_image.shape[1] < target_width:
            current_width = processed_image.shape[1]
            padding_needed = target_width - current_width
            left_padding = padding_needed // 2
            right_padding = padding_needed - left_padding
            
            # Thêm padding màu đen vào 2 bên
            processed_image = cv2.copyMakeBorder(
                processed_image, 
                0, 0,  # top, bottom
                left_padding, right_padding,  # left, right
                cv2.BORDER_CONSTANT, 
                value=[0, 0, 0]  # màu đen
            )
        
        return processed_image

    def _get_max_width_in_batch(self, images, scale_factor=2.0):
        """Tính chiều rộng tối đa sau khi upscale trong batch"""
        max_width = 0
        for img in images:
            if img is not None:
                scaled_width = int(img.shape[1] * scale_factor)
                max_width = max(max_width, scaled_width)
        return max_width
    
    def _save_paths_to_file(self, file_path: Optional[str], paths: List[str]):
        """Hàm trợ giúp để lưu một danh sách các đường dẫn ra file."""
        if file_path:
            if paths:
                try:
                    absolute_output_path = os.path.abspath(file_path)
                    print(f"\n💾 Đang lưu {len(paths)} đường dẫn vào file: {absolute_output_path}")
                    with open(file_path, 'w', encoding='utf-8') as f:
                        for path in paths:
                            f.write(f"{path}\n")
                    print("   Lưu file thành công.")
                except Exception as e:
                    print(f"   💥 Lỗi: Không thể lưu file vào {file_path}. Lý do: {e}")
            else:
                print(f"\nℹ️ Không có đường dẫn nào được tìm thấy để lưu vào file '{file_path}'.")

    def _save_paths_to_pkl_file(self, file_path: Optional[str], paths: List[str]):
        """Hàm trợ giúp để lưu một danh sách các đường dẫn ra file pickle (.pkl)."""
        if file_path:
            if paths:
                try:
                    # Đảm bảo file có đuôi .pkl để dễ nhận biết
                    if not file_path.endswith('.pkl'):
                        file_path += '.pkl'
    
                    absolute_output_path = os.path.abspath(file_path)
                    print(f"\n💾 Đang lưu {len(paths)} đường dẫn vào file pickle: {absolute_output_path}")
    
                    # Mở file ở chế độ 'write binary' (wb)
                    with open(file_path, 'wb') as f:
                        # Dùng pickle.dump để lưu toàn bộ đối tượng list vào file
                        pickle.dump(paths, f)
                    print("   Lưu file pickle thành công.")
    
                except Exception as e:
                    print(f"   💥 Lỗi: Không thể lưu file pickle vào {file_path}. Lý do: {e}")
            else:
                print(f"\nℹ️ Không có đường dẫn nào được tìm thấy để lưu vào file pickle '{file_path}'.")

    # --- PHƯƠNG THỨC CHÍNH ĐÃ ĐƯỢC CẬP NHẬT HOÀN CHỈNH ---
    def classify_images(self, 
                        batch_size: int = 64, 
                        with_text_file: Optional[str] = "paths_with_text.pkl", 
                        without_text_file: Optional[str] = "paths_without_text.pkl"
                        ) -> Tuple[List[str], List[str]]:
        """
        Phân loại ảnh thành nhóm có và không có văn bản, lưu kết quả ra file,
        và trả về cả hai danh sách.

        Args:
            batch_size (int): Số lượng ảnh xử lý trong mỗi batch.
            with_text_file (Optional[str]): Tên file để lưu các đường dẫn có văn bản.
            without_text_file (Optional[str]): Tên file để lưu các đường dẫn không có văn bản.

        Returns:
            Tuple[List[str], List[str]]: Một tuple chứa (danh_sách_có_text, danh_sách_không_có_text).
        """
        if not self.paths_to_process:
            print("Không có ảnh nào trong phạm vi được chọn để xử lý.")
            return [], []

        print(f"Bắt đầu phân loại {len(self.paths_to_process)} ảnh...")

        failed_read_paths = []
        paths_with_text = []
        paths_without_text = []

        for i in tqdm(range(0, len(self.paths_to_process), batch_size), desc="Tổng tiến trình"):
            batch_paths = self.paths_to_process[i:i + batch_size]
            
            batch_read_results = [self._read_image_and_path(path) for path in batch_paths]
            valid_images_with_paths = [(img, path) for img, path in batch_read_results if img is not None]
            failed_paths_in_batch = [path for img, path in batch_read_results if img is None]
            failed_read_paths.extend(failed_paths_in_batch)
            if not valid_images_with_paths: 
                continue
            batch_images, current_paths = zip(*valid_images_with_paths)
            
            # Tính chiều rộng tối đa trong batch sau khi upscale
            max_width = self._get_max_width_in_batch(batch_images)
            
            # Preprocess với padding để có cùng chiều rộng
            processed_images = [self._preprocess_image(img, target_width=max_width) for img in batch_images]
            processed_images = [img for img in processed_images if img is not None]
            
            if not processed_images: 
                paths_without_text.extend(current_paths)
                continue
                
            detection_results = self.reader.detect(np.array(processed_images), reformat=False)
            # --- LOGIC PHÂN LOẠI CỐT LÕI ---
            for idx, (h_boxes, f_boxes) in enumerate(zip(*detection_results)):
                if h_boxes or f_boxes:
                    paths_with_text.append(current_paths[idx])
                else:
                    paths_without_text.append(current_paths[idx])
        
        print("\n✅ Hoàn tất việc phân loại ảnh.")
        if failed_read_paths:
            print(f"⚠️ Ghi nhận {len(failed_read_paths)} ảnh bị lỗi khi đọc.")
        
        # --- LƯU CẢ HAI FILE KẾT QUẢ ---
        self._save_paths_to_file(with_text_file.replace('.pkl', '.txt'), paths_with_text)
        self._save_paths_to_file(without_text_file.replace('.pkl', '.txt'), paths_without_text)

        self._save_paths_to_pkl_file(with_text_file, paths_with_text)
        self._save_paths_to_pkl_file(without_text_file, paths_without_text)
        # --- TRẢ VỀ CẢ HAI DANH SÁCH ---
        return paths_with_text, paths_without_text

In [ ]:
DATASET_PATH = "/kaggle/input/lucifer-kfn-1/_output_/lucifer-kfn1"
# logo_box = ((520, 40), (595, 75))
# subtitle_box = ((0, 435), (640, 460))
logo_box = ((437, 14), (505, 47))
logo2_box = ((3, 2), (55, 34))
subtitle_box = ((0, 272), (533, 288))
boxes_to_ignore = [logo_box, logo2_box, subtitle_box]
# progress = {'length': 139457, 'start': 0, 'end': 1000}
progress = {}
classifier = ImageClassifier(input_path=DATASET_PATH, ignore_boxes=boxes_to_ignore, gpu=True, progress=progress)
with_text, without_text = classifier.classify_images(batch_size=64)

print(f"image ratio with text: {len(with_text) / (len(with_text) + len(without_text))}")

Đang quét và sắp xếp tất cả các đường dẫn ảnh...
{'length': 139457, 'start': 0, 'end': 1000}
Đã tìm thấy tổng cộng 139457 ảnh.
Sẽ xử lý 1000 ảnh từ chỉ số 0 đến 1000.
Đang khởi tạo EasyOCR Reader...
EasyOCR Reader đã sẵn sàng (GPU: True).
Bắt đầu phân loại 1000 ảnh...


Tổng tiến trình:   0%|          | 0/16 [00:00<?, ?it/s]


✅ Hoàn tất việc phân loại ảnh.

💾 Đang lưu 305 đường dẫn vào file: /kaggle/working/paths_with_text.txt
   Lưu file thành công.

💾 Đang lưu 695 đường dẫn vào file: /kaggle/working/paths_without_text.txt
   Lưu file thành công.

💾 Đang lưu 305 đường dẫn vào file pickle: /kaggle/working/paths_with_text.pkl
   Lưu file pickle thành công.

💾 Đang lưu 695 đường dẫn vào file pickle: /kaggle/working/paths_without_text.pkl
   Lưu file pickle thành công.
image ratio with text: 0.305
